# RiskDrift — End-to-End Pipeline Exploration

This notebook demonstrates the complete RiskDrift pipeline:
1. [Optional] Download 10-K filings from SEC EDGAR
2. Extract Item 1A text
3. Generate FinBERT embeddings
4. Compute drift scores and z-score anomaly flags
5. Visualise results
6. Backtest the long-short signal

**Quick start (no download required):** Cells marked `[SAMPLE]` use the pre-processed data in `data/sample/` and can be run immediately without downloading raw filings.

In [ ]:
import sys
sys.path.insert(0, '..')  # ensure src/ is importable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

print('Setup complete.')

## 1. [OPTIONAL] Download Filings

Skip this cell to use the sample dataset. Requires SEC EDGAR access and ~30 min for 10 tickers × 10 years.

In [ ]:
# OPTIONAL — uncomment to download
# from src.pipeline.edgar_downloader import batch_download_10k
# TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'BA', 'GE', 'AMZN', 'META', 'NFLX']
# batch_download_10k(TICKERS, start_year=2015, end_year=2023)

## 2. [OPTIONAL] Extract Item 1A Text

In [ ]:
# OPTIONAL — uncomment to extract
# from src.pipeline.extractor import process_all
# TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'BA', 'GE', 'AMZN', 'META', 'NFLX']
# process_all(TICKERS)

## 3. [OPTIONAL] Generate FinBERT Embeddings

Requires PyTorch. GPU recommended but not required (~2 min/ticker on CPU).

In [ ]:
# OPTIONAL — uncomment to embed
# from src.pipeline.embedder import embed_all
# embed_all(['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'BA', 'GE', 'AMZN', 'META', 'NFLX'])

## 4. [SAMPLE] Load Drift Scores

Uses pre-computed scores from `data/sample/drift_scores_sample.csv`.

In [ ]:
scores = pd.read_csv('../data/sample/drift_scores_sample.csv')
scores['drift_flag'] = scores['drift_flag'].str.strip().map({'True': True, 'False': False, True: True, False: False})
scores['year'] = scores['year'].astype(int)

print(f'Loaded {len(scores)} observations across {scores["ticker"].nunique()} tickers')
scores.head(10)

## 5. Drift Flag Summary

In [ ]:
flagged = scores[scores['drift_flag'] == True].copy()
print(f'Total drift flags: {len(flagged)}')
print(f'Flagged tickers: {sorted(flagged["ticker"].unique())}')
flagged[['ticker', 'year', 'cosine_similarity', 'z_score', 'sector']].sort_values('z_score')

## 6. Drift Timeline: Boeing (BA) — 737 MAX Case Study

Boeing is an informative validation case: the 737 MAX grounding began in March 2019, and the 2019 10-K (filed Feb 2020) should show a dramatic revision to Item 1A risk factors.

In [ ]:
ba = scores[scores['ticker'] == 'BA'].dropna(subset=['z_score'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ba['year'], y=ba['cosine_similarity'],
    mode='lines+markers', name='Cosine Similarity',
    line={'color': '#1f77b4'}
))

# Highlight 2019 flag
ba_flagged = ba[ba['drift_flag'] == True]
fig.add_trace(go.Scatter(
    x=ba_flagged['year'], y=ba_flagged['cosine_similarity'],
    mode='markers', name='Drift Flag',
    marker={'symbol': 'x', 'size': 16, 'color': 'red'}
))

fig.update_layout(
    title='Boeing (BA) — Item 1A Cosine Similarity (737 MAX crisis visible in 2019)',
    xaxis_title='Fiscal Year',
    yaxis_title='Cosine Similarity',
    yaxis={'range': [0.6, 1.0]},
    height=400
)
fig.show()

## 7. Sector Heatmap

In [ ]:
from src.analysis.sector_aggregator import sector_drift_heatmap

heatmap = sector_drift_heatmap(scores, metric='cosine_similarity')

fig = px.imshow(
    heatmap,
    color_continuous_scale='RdYlGn',
    title='Mean Item 1A Cosine Similarity by Sector and Year',
    labels={'color': 'Mean Cosine Similarity'},
    aspect='auto'
)
fig.update_layout(height=450)
fig.show()

## 8. Z-Score Distribution

All z-scores should be roughly normally distributed. Very negative tails indicate unusual risk language revisions.

In [ ]:
import plotly.figure_factory as ff

z_vals = scores['z_score'].dropna().tolist()
fig = ff.create_distplot([z_vals], ['Z-Score'], bin_size=0.3, show_rug=False)
fig.add_vline(x=-2.0, line_dash='dash', line_color='red', annotation_text='Threshold (-2.0)')
fig.update_layout(title='Distribution of Drift Z-Scores', xaxis_title='Z-Score', height=400)
fig.show()

print(f'Mean: {pd.Series(z_vals).mean():.3f}')
print(f'Std:  {pd.Series(z_vals).std():.3f}')
print(f'Flag rate: {(pd.Series(z_vals) < -2.0).mean():.1%}')

## 9. [OPTIONAL] Run Backtest

Requires yfinance and internet access. Fetches 6-month forward returns for each flagged company.

In [ ]:
# OPTIONAL — uncomment to run backtest
# from src.analysis.backtest import fetch_forward_returns, run_backtest, print_backtest_report
#
# Approximate 10-K filing dates (60 days after fiscal year end)
# filing_dates = {
#     'BA':  {2019: '2020-02-03', 2020: '2021-02-01'},
#     'AAPL': {2019: '2019-10-31', 2020: '2020-10-29'},
#     # ... add others
# }
# tickers = scores['ticker'].unique().tolist()
# returns_df = fetch_forward_returns(tickers, filing_dates)
# metrics = run_backtest(scores, returns_df)
# print_backtest_report(metrics)